In [ ]:
import numpy as np
import os
import rasterio

In [2]:
dataset_path = "./berlin-urban-gradient/"
tif_path = dataset_path + "raster/hymap_berlin.tif"
out_dir = dataset_path + "patches/"

patch_width = 80
patch_height = 80

In [3]:
# load tile
dataset = rasterio.open(tif_path)
data = dataset.read()
# crop
data_cropped = data[:, :, 13:-13]
# min-max normalization to range [0,1]
val_min = data_cropped.min()
val_max = data_cropped.max()
data_cropped_normalized = (data_cropped - val_min) / (val_max - val_min)
# float64 -> float32
data_cropped_normalized_float32 = np.float32(data_cropped_normalized)

In [4]:
# split into non-overlapping patches
C, H, W = data_cropped_normalized_float32.shape
patches = data_cropped_normalized_float32.reshape(
    C,
    H // patch_height, patch_height,
    W // patch_width, patch_width
).transpose(1, 3, 0, 2, 4).reshape(-1, C, patch_height, patch_width)

In [5]:
# create directory to save patches
os.makedirs(out_dir, exist_ok=True)
# save each patch as .npy with zero-padded numbering
for i, patch in enumerate(patches):
    filename = os.path.join(out_dir, f"{i:03d}.npy")  # 000.npy, 001.npy, ...
    np.save(filename, patch)

print(f"Saved {len(patches)} patches to {out_dir}")

Saved 160 patches to ./berlin-urban-gradient/patches/
